# Day 15 / 42: Decision Trees
### 42 Days of ML Challenge | @VaishnaviJagtap18

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week3_core_ml/day15_decision_trees/day15_notebook.ipynb)

---

## What You Will Learn
- Gini impurity and entropy: what they measure, computed by hand
- How a decision tree picks WHERE to split — reproduced manually and matched against sklearn
- Build, visualize, and read a real decision tree
- Why depth=10 gets 100% training accuracy and 63.7% test accuracy on the same data
- Why insurance companies use decision trees because regulators demand explainability

---

## Step 0: Install and Import

In [ ]:
!pip install numpy pandas matplotlib scikit-learn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("All imports successful. You are ready for Day 15.")

---
## Step 1: Gini Impurity and Entropy — What They Actually Measure

A decision tree builds itself by repeatedly asking: "Which question splits this group into the PUREST possible subgroups?"

"Pure" means a group where everyone has the same label. "Impure" means a 50/50 mix — maximum confusion.

**Gini Impurity:**
$$Gini = 1 - p^2 - (1-p)^2$$

**Entropy:**
$$Entropy = -p \log_2(p) - (1-p)\log_2(1-p)$$

Where `p` is the fraction of the group belonging to class 1.

Both measure the same idea (impurity) on slightly different scales. sklearn defaults to Gini because it's faster to compute (no logarithms).

In [ ]:
def gini_impurity(p):
    return 1 - p**2 - (1-p)**2

def entropy(p):
    if p == 0 or p == 1:
        return 0
    return -p*np.log2(p) - (1-p)*np.log2(1-p)

print(f"{'p (fraction class 1)':>22} | {'Gini':>8} | {'Entropy':>8}")
print("-" * 46)
for p in [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
    print(f"{p:>22.1f} | {gini_impurity(p):>8.4f} | {entropy(p):>8.4f}")

print()
print("p=0.0 or p=1.0 (pure group, everyone same label): impurity = 0")
print("p=0.5 (perfect 50/50 mix, maximum confusion): impurity is at its HIGHEST")
print()
print("A decision tree's job: find splits that move groups AWAY from p=0.5")
print("and TOWARD p=0 or p=1 — i.e. reduce impurity as much as possible.")

p_range = np.linspace(0, 1, 100)
plt.figure(figsize=(7,5))
plt.plot(p_range, [gini_impurity(p) for p in p_range], label='Gini', linewidth=2.5, color='steelblue')
plt.plot(p_range, [entropy(p) for p in p_range], label='Entropy', linewidth=2.5, color='darkorange')
plt.xlabel('p (fraction belonging to class 1)')
plt.ylabel('Impurity')
plt.title('Gini Impurity vs Entropy')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('day15_gini_entropy.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Step 2: How a Split Is Chosen — By Hand

Tiny dataset: 8 loan applicants, their credit score, and whether they were approved.

The tree tries every possible threshold on credit_score, computes the **weighted Gini** of the two resulting groups, and picks the threshold that gives the LOWEST weighted Gini (the purest split).

$$\text{Weighted Gini} = \frac{n_{left}}{n_{total}}Gini_{left} + \frac{n_{right}}{n_{total}}Gini_{right}$$

In [ ]:
credit_scores = np.array([580, 620, 650, 680, 700, 720, 750, 800])
approved      = np.array([0,    0,   1,   0,   1,   1,   1,   1])

parent_gini = gini_impurity(approved.mean())
print(f"Parent node gini (before any split): {parent_gini:.4f}")
print(f"  ({approved.sum()} approved out of {len(approved)} -> p={approved.mean():.3f})")
print()

candidate_thresholds = [620, 650, 680, 700]

print(f"{'Threshold':>10} | {'Left group':>22} | {'Right group':>18} | {'Weighted Gini':>14}")
print("-" * 72)

best_threshold = None
best_gini = 999

for threshold in candidate_thresholds:
    left_mask  = credit_scores <= threshold
    right_mask = ~left_mask

    left_labels  = approved[left_mask]
    right_labels = approved[right_mask]

    left_gini  = gini_impurity(left_labels.mean())
    right_gini = gini_impurity(right_labels.mean())

    n_left, n_right = len(left_labels), len(right_labels)
    n_total = n_left + n_right

    weighted = (n_left/n_total)*left_gini + (n_right/n_total)*right_gini

    print(f"<= {threshold:>7} | {str(left_labels):>22} | {str(right_labels):>18} | {weighted:>14.4f}")

    if weighted < best_gini:
        best_gini = weighted
        best_threshold = threshold

print()
print(f"Best split: credit_score <= {best_threshold}  (weighted gini = {best_gini:.4f})")
print(f"Impurity reduction from parent: {parent_gini:.4f} -> {best_gini:.4f}")
print()

# Verify with sklearn
X_tiny = credit_scores.reshape(-1, 1)
tree_tiny = DecisionTreeClassifier(max_depth=1, criterion='gini', random_state=42).fit(X_tiny, approved)
sklearn_threshold = tree_tiny.tree_.threshold[0]
print(f"sklearn's chosen threshold for this exact data: {sklearn_threshold}")
print(f"Our manual best ({best_threshold}) and sklearn's ({sklearn_threshold}) point to the")
print("same split region (680-700) — sklearn just searches more candidate points.")

---
## Step 3: A Real Dataset — Loan Approval (With Realistic Noise)

Real data is never perfectly clean. 15% of labels here are randomly flipped — simulating human error, edge cases, and exceptions that don't follow the "rule."

This noise is what makes the depth experiment in Step 5 meaningful.

In [ ]:
np.random.seed(42)
n = 400

credit_score = np.random.randint(300, 850, n).astype(float)
income       = np.random.randint(20000, 150000, n).astype(float)
debt_ratio   = np.random.uniform(0, 1, n)

# Underlying rule: good credit AND low debt ratio -> approved
base_approved = ((credit_score > 650) & (debt_ratio < 0.4)).astype(int)

# Add 15% label noise — real-world exceptions and edge cases
noise_mask = np.random.rand(n) < 0.15
approved = np.where(noise_mask, 1 - base_approved, base_approved)

df = pd.DataFrame({
    'credit_score': credit_score, 'income': income,
    'debt_ratio': debt_ratio, 'approved': approved
})

print("Sample data:")
print(df.head(8).round(2).to_string(index=False))
print(f"\nDataset size: {n}")
print(f"Approval rate: {approved.mean()*100:.1f}%")
print(f"Labels affected by noise: {noise_mask.sum()} out of {n} ({noise_mask.mean()*100:.0f}%)")

---
## Step 4: Build and Visualize a Tree (max_depth=3)

In [ ]:
X = df[['credit_score', 'income', 'debt_ratio']].values
y = df['approved'].values
feature_names = ['credit_score', 'income', 'debt_ratio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree3 = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
tree3.fit(X_train, y_train)

train_acc = tree3.score(X_train, y_train)
test_acc  = tree3.score(X_test, y_test)

print(f"max_depth=3 -> train accuracy: {train_acc:.4f}, test accuracy: {test_acc:.4f}")
print()
print("=== Feature Importances ===")
for name, imp in zip(feature_names, tree3.feature_importances_):
    print(f"  {name:>15}: {imp:.4f}")
print()
print("=== Reading the Tree (text form) ===")
print(export_text(tree3, feature_names=feature_names))

plt.figure(figsize=(16, 8))
plot_tree(tree3, feature_names=feature_names, class_names=['Rejected','Approved'],
          filled=True, fontsize=9, rounded=True)
plt.title('Decision Tree (max_depth=3) — Loan Approval')
plt.tight_layout()
plt.savefig('day15_tree_depth3.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print("Each box shows: the question asked, the gini value (impurity AFTER this split),")
print("the number of samples, and the class distribution. Darker color = purer node.")

---
## Step 5: The Depth Experiment — Overfitting in Real Numbers

An unconstrained decision tree (no `max_depth`) will keep splitting until every leaf is 100% pure — even if that means creating a leaf for a single noisy data point.

**This is overfitting**: the tree memorizes the training data, including its noise, instead of learning the underlying pattern.

Watch what happens to train vs test accuracy as depth increases.

In [ ]:
depths_to_try = [1, 2, 3, 5, 7, 10, None]

print(f"{'max_depth':>10} | {'actual depth':>12} | {'train acc':>10} | {'test acc':>9} | {'gap':>8}")
print("-" * 58)

results = []
for depth in depths_to_try:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    tr_acc = t.score(X_train, y_train)
    te_acc = t.score(X_test, y_test)
    gap = tr_acc - te_acc
    results.append((depth, t.get_depth(), tr_acc, te_acc, gap))
    depth_label = 'None' if depth is None else str(depth)
    print(f"{depth_label:>10} | {t.get_depth():>12} | {tr_acc:>10.4f} | {te_acc:>9.4f} | {gap:>8.4f}")

print()
print(f"At depth=10, train accuracy hits {results[5][2]*100:.1f}% — the tree memorized the")
print(f"training data, including the 15% label noise.")
print(f"Test accuracy drops to {results[5][3]*100:.1f}% — a gap of {results[5][4]*100:.1f} percentage points.")
print()
print(f"At depth=3, the gap is only {abs(results[2][4])*100:.1f} points — and test accuracy")
print("is actually as good as or better than the deeper trees.")

# Plot
depth_labels = [str(d) if d is not None else 'None' for d, _, _, _, _ in results]
train_accs = [r[2] for r in results]
test_accs  = [r[3] for r in results]

plt.figure(figsize=(8,5))
x_pos = range(len(depth_labels))
plt.plot(x_pos, train_accs, 'o-', label='Train Accuracy', color='steelblue', linewidth=2)
plt.plot(x_pos, test_accs, 'o-', label='Test Accuracy', color='darkorange', linewidth=2)
plt.xticks(x_pos, depth_labels)
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Overfitting: Train vs Test Accuracy by Tree Depth')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('day15_depth_overfitting.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Step 6: The Real-World Production Problem

**The scenario:** An insurance company builds a claims approval model. The data scientist trains a decision tree with no `max_depth` set (sklearn's default). Training accuracy: 100%. The model ships.

Three months later, the model's real-world accuracy is far lower than the reported 100%, and the compliance team asks: "Why did the model deny this specific claim?" The answer is a 47-level-deep tree with thousands of leaves — completely unreadable.

**Two separate problems, one root cause (unconstrained depth):**

1. **Overfitting**: 100% training accuracy was never real performance — it was memorization of training data, including data entry errors and one-off exceptions.

2. **Explainability**: Insurance regulators require that a denial can be explained in plain terms ("denied because claim amount exceeded policy limit AND prior claims in the last 12 months"). A tree with depth 47 cannot produce that explanation — there's no human-readable path.

**The fix:** Constrain `max_depth` (commonly 3-6 for regulated use cases). A shallow tree trades a small amount of raw accuracy for two things regulated industries need: a model that generalizes to new claims, and a decision path a human can read and explain.

This is exactly why decision trees remain popular in insurance, credit scoring, and healthcare — not despite being "simple," but because the simplicity IS the requirement.

In [ ]:
# Demonstrate: explain ONE prediction using the shallow tree
sample_idx = 0
sample = X_test[sample_idx].reshape(1, -1)
actual = y_test[sample_idx]
pred = tree3.predict(sample)[0]

print("=== Explaining a single prediction (max_depth=3 tree) ===")
print(f"Applicant: credit_score={sample[0][0]:.0f}, income={sample[0][1]:.0f}, debt_ratio={sample[0][2]:.2f}")
print(f"Actual outcome: {'Approved' if actual==1 else 'Rejected'}")
print(f"Model prediction: {'Approved' if pred==1 else 'Rejected'}")
print()

# Walk the decision path
node_indicator = tree3.decision_path(sample)
leaf_id = tree3.apply(sample)
feature = tree3.tree_.feature
threshold = tree3.tree_.threshold

node_index = node_indicator.indices[node_indicator.indptr[0]:node_indicator.indptr[1]]

print("Decision path:")
for node_id in node_index:
    if leaf_id[0] == node_id:
        print(f"  -> Leaf reached: predict {'Approved' if pred==1 else 'Rejected'}")
        continue
    f_name = feature_names[feature[node_id]]
    f_val = sample[0][feature[node_id]]
    thresh = threshold[node_id]
    direction = "<=" if f_val <= thresh else ">"
    print(f"  Node {node_id}: is {f_name} ({f_val:.2f}) <= {thresh:.2f}?  -> {direction} {thresh:.2f}")

print()
print("This is a human-readable explanation a compliance officer can verify.")
print("This is impossible to produce cleanly for a depth-47 tree.")

---
## Step 7: Summary — Decision Tree Rules

In [ ]:
print("=" * 62)
print("DAY 15 SUMMARY: Decision Trees")
print("=" * 62)
print()
print("HOW SPLITS WORK")
print("-" * 50)
print("1. Gini/Entropy measure impurity: 0 = pure group, max at 50/50")
print("2. The tree tries many thresholds per feature, picks the one")
print("   with the LOWEST weighted impurity after the split")
print("3. This repeats recursively for every new node (greedy algorithm)")
print()
print("DEPTH AND OVERFITTING")
print("-" * 50)
print("4. No max_depth -> tree grows until every leaf is pure")
print("5. This means memorizing training data, including noise")
print("6. 100% train accuracy is a WARNING SIGN, not a success")
print("7. Shallow trees (depth 3-6) often generalize BETTER, not worse")
print()
print("WHY TREES ARE USED IN REGULATED INDUSTRIES")
print("-" * 50)
print("8. Every prediction maps to a readable if/then path")
print("9. Insurance, credit scoring, healthcare often legally REQUIRE this")
print("10. Feature importance shows which inputs drove the splits most")
print()
print("=" * 62)

---
## Practice Exercise

A customer churn dataset is given below.

Your tasks:
1. Train decision trees at `max_depth` = 2, 4, 8, and `None`
2. Print train and test accuracy for each
3. Identify where overfitting begins (where the train/test gap grows)
4. Pick the best `max_depth` and justify your choice in one sentence
5. Print the feature importances for your chosen tree

In [ ]:
# Practice dataset — customer churn
np.random.seed(33)
n_practice = 500

tenure_months   = np.random.randint(1, 72, n_practice).astype(float)
monthly_charges = np.random.uniform(20, 120, n_practice)
support_tickets = np.random.randint(0, 10, n_practice).astype(float)

base_churn = ((tenure_months < 12) & (support_tickets > 4)).astype(int)
noise_mask_p = np.random.rand(n_practice) < 0.12
churned = np.where(noise_mask_p, 1 - base_churn, base_churn)

practice_df = pd.DataFrame({
    'tenure_months': tenure_months,
    'monthly_charges': monthly_charges,
    'support_tickets': support_tickets,
    'churned': churned
})

print("Practice dataset (customer churn):")
print(practice_df.head(8).round(2).to_string(index=False))
print(f"\nShape: {practice_df.shape}")
print(f"Churn rate: {churned.mean()*100:.1f}%")
print()
print("Your tasks:")
print("  1. Train trees at max_depth = 2, 4, 8, None")
print("  2. Print train/test accuracy for each")
print("  3. Where does overfitting begin?")
print("  4. Pick the best max_depth and justify it")
print("  5. Print feature importances for your chosen tree")

# --- Your solution below ---


In [ ]:
# SOLUTION — try on your own first before looking here

X_churn = practice_df[['tenure_months','monthly_charges','support_tickets']].values
y_churn = practice_df['churned'].values
feat_names_churn = ['tenure_months','monthly_charges','support_tickets']

Xtr, Xte, ytr, yte = train_test_split(X_churn, y_churn, test_size=0.2, random_state=42)

print(f"{'max_depth':>10} | {'train acc':>10} | {'test acc':>9} | {'gap':>8}")
print("-" * 46)
for depth in [2, 4, 8, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(Xtr, ytr)
    tr, te = t.score(Xtr,ytr), t.score(Xte,yte)
    label = 'None' if depth is None else str(depth)
    print(f"{label:>10} | {tr:>10.4f} | {te:>9.4f} | {tr-te:>8.4f}")

print()
print("Overfitting begins where the gap grows sharply (typically depth >= 8).")
print("depth=4 is a reasonable choice: small gap, test accuracy near its peak.")

best_tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xtr, ytr)
print("\nFeature importances (max_depth=4):")
for name, imp in zip(feat_names_churn, best_tree.feature_importances_):
    print(f"  {name:>16}: {imp:.4f}")

---
## What's Next

**Day 16: Random Forests**  
Bagging, why averaging many weak trees beats one strong one, and how Kaggle competition winners use Random Forest as a baseline that beats most teams.

---
**GitHub repo:** https://github.com/VaishnaviJagtap18/42-days-aiml-challenge  
**LinkedIn:** Follow for Day 16 tomorrow  
#42DaysOfML #MachineLearning #Python #MLEngineer